In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../notebooks/lakehouse.duckdb")
con.execute("SHOW TABLES").df()

,name
0,dim_cliente
1,dim_producto
2,fact_cliente
3,fact_producto
4,raw_customers
5,raw_orders
6,raw_products
7,stage_customers
8,stage_orders
9,stage_products


In [3]:
# Producto con mas ingresos totales por categoria de producto
con.execute("select * from fact_producto limit 5").df()

,product_id,unidades_vendidas,ingresos_totales,num_pedidos
0,1,22.0,4148.5,6
1,4,7.0,99.4,3
2,5,4.0,49.9,2
3,6,8.0,519.9,2
4,7,4.0,118.9,2


In [4]:
con.execute("SELECT * from dim_producto limit 5").df()

,product_id,product_name,category,price
0,1,PRODUCTO 1,DEPORTES,5.0
1,4,PRODUCTO 4,BELLEZA,120.0
2,5,PRODUCTO 5,HOGAR,19.9
3,6,PRODUCTO 6,ELECTRONICA,49.5
4,7,PRODUCTO 7,BELLEZA,49.5


In [6]:
# Realizamos el join para esas tablas
con.execute("""
    select p.category, p.product_name, f.ingresos_totales 
    from fact_producto f 
    join dim_producto p on f.product_id = p.product_id""").df()

,category,product_name,ingresos_totales
0,DEPORTES,PRODUCTO 1,4148.5
1,BELLEZA,PRODUCTO 4,99.4
2,HOGAR,PRODUCTO 5,49.9
3,ELECTRONICA,PRODUCTO 6,519.9
4,BELLEZA,PRODUCTO 7,118.9
5,DEPORTES,PRODUCTO 9,1549.9
6,ROPA,PRODUCTO 11,99.0
7,DEPORTES,PRODUCTO 12,1519.9
8,ELECTRONICA,PRODUCTO 13,39.8
9,BELLEZA,PRODUCTO 14,1569.4


In [7]:
con.execute("""
    SELECT 
        p.category, 
        p.product_name, 
        f.ingresos_totales,
        ROW_NUMBER() OVER (PARTITION BY p.category ORDER BY f.ingresos_totales DESC) AS rn
    FROM fact_producto f
    JOIN dim_producto p ON f.product_id = p.product_id
""").df()

,category,product_name,ingresos_totales,rn
0,BELLEZA,PRODUCTO 14,1569.4,1
1,BELLEZA,PRODUCTO 7,118.9,2
2,BELLEZA,PRODUCTO 4,99.4,3
3,HOGAR,PRODUCTO 18,1519.9,1
4,HOGAR,PRODUCTO 5,49.9,2
5,ROPA,PRODUCTO 11,99.0,1
6,DEPORTES,PRODUCTO 1,4148.5,1
7,DEPORTES,PRODUCTO 16,3030.0,2
8,DEPORTES,PRODUCTO 15,1648.5,3
9,DEPORTES,PRODUCTO 9,1549.9,4


In [8]:
con.execute("""
    SELECT category, product_name, ingresos_totales
    FROM (
        SELECT 
            p.category, 
            p.product_name, 
            f.ingresos_totales,
            ROW_NUMBER() OVER (PARTITION BY p.category ORDER BY f.ingresos_totales DESC) AS rn
        FROM fact_producto f
        JOIN dim_producto p ON f.product_id = p.product_id
    ) ranked
    WHERE rn = 1
""").df()

,category,product_name,ingresos_totales
0,ELECTRONICA,PRODUCTO 6,519.9
1,HOGAR,PRODUCTO 18,1519.9
2,ROPA,PRODUCTO 11,99.0
3,DEPORTES,PRODUCTO 1,4148.5
4,BELLEZA,PRODUCTO 14,1569.4


In [9]:
con.execute("""
    SELECT category, MAX(ingresos_totales) AS max_ingresos
    FROM fact_producto f2
    JOIN dim_producto p2 ON f2.product_id = p2.product_id
    GROUP BY category
""").df()

,category,max_ingresos
0,DEPORTES,4148.5
1,ROPA,99.0
2,ELECTRONICA,519.9
3,BELLEZA,1569.4
4,HOGAR,1519.9


In [10]:
con.execute("SELECT * from dim_cliente limit 5").df()

,customer_id,full_name,email,city,age,registration_date
0,1,MARIA LOPEZ,maria1@gmail.com,LIMA,22,2023-02-02
1,2,ANA TORRES,ana2@gmail.com,TRUJILLO,19,2024-04-04
2,3,JUAN PEREZ,juan3@gmail.com,TRUJILLO,41,2024-01-25
3,4,SOFIA MENDOZA,sofia4@gmail.com,TRUJILLO,33,2023-03-22
4,5,CAMILA VARGAS,camila5@gmail.com,CUSCO,<NA>,2025-06-06


In [11]:
con.execute("SELECT * from fact_cliente limit 5").df()

,customer_id,total_pedidos,monto_total_gastado,ticket_promedio,fecha_ultimo_pedido
0,1,0,0.0,0.00,NaT
1,2,0,0.0,0.00,NaT
2,3,0,0.0,0.00,NaT
3,4,4,668.4,167.10,2025-02-19
4,5,2,79.5,39.75,2025-05-01


In [18]:
con.execute("""
    SELECT city, full_name, monto_total_gastado
    FROM (
        SELECT 
            d.city,
            d.full_name,
            f.monto_total_gastado,
            ROW_NUMBER() OVER (PARTITION BY d.city ORDER BY f.monto_total_gastado DESC) AS rn
        FROM dim_cliente d 
        JOIN fact_cliente f ON d.customer_id = f.customer_id
    ) temp
    WHERE rn = 1
""").df()

,city,full_name,monto_total_gastado
0,PIURA,MARIA LOPEZ,1500.0
1,None,DIEGO FERNANDEZ,1519.9
2,AREQUIPA,CAMILA VARGAS,2000.0
3,CUSCO,LUIS GARCIA,3099.0
4,LIMA,LUIS GARCIA,1579.5
5,TRUJILLO,DIEGO FERNANDEZ,1500.0


In [20]:
con.execute("SELECT * FROM fact_cliente limit 5").df()

,customer_id,total_pedidos,monto_total_gastado,ticket_promedio,fecha_ultimo_pedido
0,1,0,0.0,0.00,NaT
1,2,0,0.0,0.00,NaT
2,3,0,0.0,0.00,NaT
3,4,4,668.4,167.10,2025-02-19
4,5,2,79.5,39.75,2025-05-01


In [21]:
con.execute("SELECT * FROM dim_cliente limit 5").df()

,customer_id,full_name,email,city,age,registration_date
0,1,MARIA LOPEZ,maria1@gmail.com,LIMA,22,2023-02-02
1,2,ANA TORRES,ana2@gmail.com,TRUJILLO,19,2024-04-04
2,3,JUAN PEREZ,juan3@gmail.com,TRUJILLO,41,2024-01-25
3,4,SOFIA MENDOZA,sofia4@gmail.com,TRUJILLO,33,2023-03-22
4,5,CAMILA VARGAS,camila5@gmail.com,CUSCO,<NA>,2025-06-06


In [26]:
con.execute("""
    SELECT AVG(total_pedidos) AS promedio_pedidos
    FROM fact_cliente
    WHERE total_pedidos > 0
""").df()

,promedio_pedidos
0,1.6875


In [28]:
# Resolucion de la seccion 5 del enunciado
# Lista los 3 clientes con mayor número de pedidos en el ultimo trimestre disponible en los datos
#Incluuir customer_id, full_name, cantidad_pedidos
con.execute("""
    SELECT 
        o.customer_id, 
        c.full_name, 
        COUNT(*) AS cantidad_pedidos
    FROM stage_orders o
    JOIN dim_cliente c ON o.customer_id = c.customer_id
    WHERE EXTRACT(year FROM o.order_date) = (SELECT EXTRACT(year FROM MAX(order_date)) FROM stage_orders)
      AND EXTRACT(quarter FROM o.order_date) = (SELECT EXTRACT(quarter FROM MAX(order_date)) FROM stage_orders)
    GROUP BY o.customer_id, c.full_name
    ORDER BY cantidad_pedidos DESC
    LIMIT 3
""").df()

,customer_id,full_name,cantidad_pedidos
0,5,CAMILA VARGAS,1
1,32,PEDRO CASTILLO,1
2,9,LUIS GARCIA,1


In [29]:
#Calcula el revenue mensual por categoría de producto. 
#Columnas esperadas: año, mes, categoria, revenue_total. 
# Ordena de mayor a menor revenue.
con.execute("""
    SELECT
        EXTRACT(year FROM o.order_date) AS año,
        EXTRACT(month FROM o.order_date) AS mes,
        p.category AS categoria,
        SUM(o.total_amount_usd) AS revenue_total
    FROM stage_orders o
    JOIN dim_producto p ON o.product_id = p.product_id
    WHERE o.order_date IS NOT NULL
    GROUP BY EXTRACT(year FROM o.order_date), EXTRACT(month FROM o.order_date), p.category
    ORDER BY revenue_total DESC
""").df()

,año,mes,categoria,revenue_total
0,2025,3,DEPORTES,2099.0
1,2024,6,DEPORTES,2030.0
2,2025,1,DEPORTES,2000.0
3,2025,5,DEPORTES,1500.0
4,2024,12,DEPORTES,1500.0
5,2025,1,BELLEZA,1500.0
6,2024,7,HOGAR,1500.0
7,2025,2,DEPORTES,1500.0
8,2024,11,DEPORTES,1049.5
9,2024,10,ELECTRONICA,500.0


In [30]:
#Identifica los pedidos cuyo total_amount_usd supere 2 desviaciones estándar del promedio. 
# Devuelve: order_id, customer_id, total_amount_usd, z_score.

con.execute("""
    SELECT 
        order_id,
        customer_id,
        total_amount_usd,
        (total_amount_usd - (SELECT AVG(total_amount_usd) FROM stage_orders)) 
            / (SELECT STDDEV(total_amount_usd) FROM stage_orders) AS z_score
    FROM stage_orders
""").df()

,order_id,customer_id,total_amount_usd,z_score
0,1,18,49.5,-0.615315
1,2,15,19.9,-0.666261
2,5,31,500.0,0.160060
3,6,35,99.0,-0.530118
4,9,37,49.5,-0.615315
5,10,12,49.5,-0.615315
6,12,31,99.0,-0.530118
7,13,5,49.5,-0.615315
8,14,40,30.0,-0.648877
9,16,39,49.5,-0.615315


In [32]:
con.execute("""
    SELECT order_id, customer_id, total_amount_usd, z_score
    FROM (
        SELECT 
            order_id,
            customer_id,
            total_amount_usd,
            (total_amount_usd - (SELECT AVG(total_amount_usd) FROM stage_orders)) 
                / (SELECT STDDEV(total_amount_usd) FROM stage_orders) AS z_score
        FROM stage_orders
    ) con_zscore
    WHERE ABS(z_score) > 1
""").df()

,order_id,customer_id,total_amount_usd,z_score
0,17,7,1500.0,1.881203
1,22,10,1500.0,1.881203
2,26,14,1500.0,1.881203
3,36,9,1500.0,1.881203
4,37,9,1500.0,1.881203
5,44,17,1500.0,1.881203
6,50,30,1500.0,1.881203
7,53,38,1500.0,1.881203
8,57,27,1500.0,1.881203
9,66,30,1500.0,1.881203
